# Twitter Bot Detection: end-to-end example

This notebook walks through the complete profile-based workflow: locate the downloaded TwiBot-20 files, load and preprocess them, train a baseline classifier, evaluate it, and optionally select features. Run the cells in order.

## Step 1: install and import dependencies

Install the package from the repository root with `pip install -e .` and install `requirements.txt` before starting the notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

from twitter_bot_detection.eda import profile_data_preprocessing
from twitter_bot_detection.feature_selection import backwards_shap_feature_selection

## Step 2: locate the downloaded dataset

Download the public TwiBot-20 data described in the README. This example expects the new-format files under `datasets/Raw/Twibot-20-new-format/`; the repository intentionally contains placeholders instead of the raw dataset.

In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / "pyproject.toml").is_file():
    repo_root = repo_root.parent

data_dir = repo_root / "datasets" / "Raw" / "Twibot-20-new-format"
required_files = [data_dir / name for name in ("edge.csv", "label.csv", "split.csv")]
missing_files = [path for path in required_files if not path.is_file()]
profile_files = sorted(data_dir.glob("node.part*.parquet"))
if missing_files or not profile_files:
    missing = [str(path) for path in [*missing_files, data_dir / "node.part*.parquet"]]
    raise FileNotFoundError("Download TwiBot-20 and add the required files: " + ", ".join(missing))

data_dir

## Step 3: load profiles, labels, splits, and network edges

In [ ]:
profile_df = pd.concat((pd.read_parquet(path) for path in profile_files), ignore_index=True)
label_df = pd.read_csv(data_dir / "label.csv")
split_df = pd.read_csv(data_dir / "split.csv")
neighbor_df = pd.read_csv(data_dir / "edge.csv")

for name, frame in {
    "profiles": profile_df,
    "labels": label_df,
    "splits": split_df,
    "edges": neighbor_df,
}.items():
    print(f"{name}: {frame.shape}")

## Step 4: preprocess profile and network features

`profile_data_preprocessing` joins the data sources and derives profile tenure, follower ratios, and text representations of follow relationships.

In [ ]:
profiles = profile_data_preprocessing(
    profile_df=profile_df,
    label_df=label_df,
    split_df=split_df,
    neighbor_df=neighbor_df,
)
profiles[["id", "label", "split", "tenure", "followers_follow_proportion"]].head()

## Step 5: create train and test matrices

This baseline uses the numeric profile features. Convert textual labels when needed, remove incomplete rows, and keep the dataset's predefined train/test split.

In [ ]:
target = "label"
if profiles[target].dtype == object:
    profiles = profiles.assign(
        target=profiles[target].str.lower().map({"human": 0, "bot": 1})
    )
else:
    profiles = profiles.assign(target=profiles[target].astype(int))

feature_columns = profiles.select_dtypes(include=np.number).columns.difference([target])
data = profiles.replace([np.inf, -np.inf], np.nan).dropna(subset=[target, "split"])
train = data.loc[data["split"].eq("train")]
test = data.loc[data["split"].eq("test")]

train_medians = train[feature_columns].median()
X_train = train[feature_columns].fillna(train_medians)
X_test = test[feature_columns].fillna(train_medians)
y_train, y_test = train[target], test[target]
X_train.shape, X_test.shape

## Step 6: train and evaluate a baseline model

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)
model.fit(X_train, y_train)
test_probabilities = model.predict_proba(X_test)[:, 1]
test_predictions = model.predict(X_test)

print(f"Test ROC AUC: {roc_auc_score(y_test, test_probabilities):.3f}")
print(classification_report(y_test, test_predictions))

## Step 7: optionally select features with SHAP

Use a validation subset from the training split to identify features that can be removed. Start with a small number of iterations while exploring the workflow; increase the limits for a full experiment.

In [ ]:
validation = train.sample(frac=0.2, random_state=42)
selection_train = train.drop(validation.index)
selection_logs = backwards_shap_feature_selection(
    model=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    df_train=selection_train.assign(**{feature: X_train.loc[selection_train.index, feature] for feature in feature_columns}),
    df_val=validation.assign(**{feature: X_train.loc[validation.index, feature] for feature in feature_columns}),
    candidate_features_for_removal=list(feature_columns),
    target=target,
    bootstrap_samples=20,
    max_iter=1,
)
selection_logs

## Next steps

Use the selected feature set to retrain the model, compare validation and test metrics, and add the repository's network and BERT content embeddings for the complete multimodal experiment.